# Kimuntu AI: Baseline Legal Query Classifier (Week 1)

This notebook trains and evaluates the first text classification component of Kimuntu AI.
It sorts an English user query into one of **9 categories** (8 legal areas + out-of-scope) and returns a
**confidence score**, so that the rest of the system (retrieval → decision support → recommendation → chatbot)
knows which area of law the user is asking about, or whether it should ask the user to clarify instead.

> **Disclaimer:** Kimuntu AI supports user understanding and does **not** replace professional legal advice.

**How to run**
- **Google Colab:** *Runtime → Run all*. When asked, upload `data/processed/dataset.csv` from the project folder.
- **Locally:** open this notebook from the `notebooks/` folder with the project's virtual environment.

The notebook is equivalent to `src/train_baseline.py` (same data, same seed, same settings), so it gives the same numbers.

## 1. Setup
We use only **scikit-learn**, **pandas**, **numpy**, **matplotlib** and **joblib**. These are pre-installed on Colab.
A fixed random seed (`SEED = 42`) makes the results reproducible.

In [ ]:
from itertools import product
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_fscore_support)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

SEED = 42
np.random.seed(SEED)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## 2. Load the prepared dataset
`dataset.csv` is produced by `src/prepare_data.py`. It combines:
- **Law Stack Exchange** questions (real, international, CC BY-SA 4.0). A question is kept only when its tags map to exactly one of our categories.
- **Synthetic** Uganda-context queries written by the project team with AI assistance. These are **clearly marked** in the `source` column.

The text is already cleaned (HTML and URLs removed, lowercased) and split 70/15/15 into train/validation/test (stratified, seed 42, duplicates removed before splitting).

In [ ]:
CANDIDATES = [Path("data/processed/dataset.csv"), Path("../data/processed/dataset.csv"), Path("dataset.csv")]
data_path = next((p for p in CANDIDATES if p.exists()), None)
if data_path is None:
    try:
        from google.colab import files  # only available on Colab
        print("Please upload data/processed/dataset.csv")
        files.upload()
        data_path = Path("dataset.csv")
    except ImportError:
        raise FileNotFoundError("dataset.csv not found. Run src/prepare_data.py first.")

df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows from {data_path}")
df.head()

In [ ]:
pd.crosstab(df["label"], [df["source"], df["split"]], margins=True)

In [ ]:
ax = df[df["split"] == "train"].groupby(["label", "source"]).size().unstack().plot.barh(stacked=True, figsize=(8, 5))
ax.set_title("Training examples per category and source")
ax.set_xlabel("number of examples")
plt.tight_layout()
plt.show()

The classes are **imbalanced** (some categories have many more Law Stack Exchange questions than others),
and the `other_out_of_scope` class comes only from synthetic data. This is why we use **macro-F1** to choose a model:
it gives every category equal weight, so the model cannot score well by only getting the large classes right.

## 3. Model selection on the validation set
Each candidate is a **pipeline**: `TfidfVectorizer` (turns text into word-weight vectors) followed by a classifier.
Because the vectoriser is inside the pipeline, it is fitted on the **training split only**. Nothing from the
validation or test text leaks into training.

Candidates:
- **Dummy (most frequent class):** a floor that any useful model must beat.
- **Multinomial Naive Bayes:** fast and simple, a classic text baseline.
- **Logistic Regression** (`class_weight="balanced"` to compensate for imbalance): interpretable weights per word, gives probabilities.

We try unigrams vs unigrams+bigrams, with/without English stop words, and a few regularisation values, then pick
the configuration with the best **validation macro-F1**. The test set is not touched here.

In [ ]:
train, val, test = (df[df["split"] == s] for s in ["train", "validation", "test"])
print(f"train={len(train)}  validation={len(val)}  test={len(test)}")

def make_pipeline(model_name, ngram_range, stop_words, param):
    tfidf = TfidfVectorizer(ngram_range=ngram_range, stop_words=stop_words, sublinear_tf=True)
    if model_name == "naive_bayes":
        clf = MultinomialNB(alpha=param)
    elif model_name == "logistic_regression":
        clf = LogisticRegression(C=param, class_weight="balanced", max_iter=3000, random_state=SEED)
    else:
        clf = DummyClassifier(strategy="most_frequent")
    return Pipeline([("tfidf", tfidf), ("clf", clf)])

def candidate_configs():
    yield {"model": "dummy_most_frequent", "ngram_range": (1, 1), "stop_words": None, "param": None}
    for ngram, stop in product([(1, 1), (1, 2)], [None, "english"]):
        for alpha in [0.01, 0.1, 0.5, 1.0]:
            yield {"model": "naive_bayes", "ngram_range": ngram, "stop_words": stop, "param": alpha}
        for c in [0.5, 1.0, 5.0, 10.0]:
            yield {"model": "logistic_regression", "ngram_range": ngram, "stop_words": stop, "param": c}

comparison = []
for cfg in candidate_configs():
    pipe = make_pipeline(cfg["model"], cfg["ngram_range"], cfg["stop_words"], cfg["param"])
    pipe.fit(train["text"], train["label"])
    pred = pipe.predict(val["text"])
    comparison.append({**cfg, "val_accuracy": accuracy_score(val["label"], pred),
                       "val_f1_macro": f1_score(val["label"], pred, average="macro", zero_division=0)})
comparison = pd.DataFrame(comparison).sort_values("val_f1_macro", ascending=False)
comparison.groupby("model").head(1)   # best configuration of each model type

In [ ]:
comparison.head(8)

In [ ]:
best = comparison.iloc[0].to_dict()
best_cfg = {k: best[k] for k in ["model", "ngram_range", "stop_words", "param"]}
best_cfg["stop_words"] = None if pd.isna(best_cfg["stop_words"]) else best_cfg["stop_words"]
print("Selected:", best_cfg)
model = make_pipeline(best_cfg["model"], best_cfg["ngram_range"], best_cfg["stop_words"], best_cfg["param"])
model.fit(train["text"], train["label"])

## 4. Confidence threshold (safety gate)
The model gives a probability for each category. The highest one is the **confidence**.
If the confidence is low, the system should **ask the user to clarify or refer them to a professional** instead of
continuing with a possibly wrong category. We choose the threshold on the **validation** set as the lowest value at which
the queries the system *does* answer are at least 90% correct (a team target, which can be changed).

In [ ]:
TARGET_ACCEPTED_ACCURACY = 0.90
THRESHOLDS = np.round(np.arange(0.20, 0.96, 0.05), 2)

def threshold_table(confidence, correct):
    rows = []
    for t in THRESHOLDS:
        accepted = confidence >= t
        rows.append({"threshold": t, "coverage": accepted.mean(),
                     "accepted_accuracy": correct[accepted].mean() if accepted.any() else np.nan})
    return pd.DataFrame(rows)

val_proba = model.predict_proba(val["text"])
val_correct = model.classes_[val_proba.argmax(axis=1)] == val["label"].to_numpy()
val_thr = threshold_table(val_proba.max(axis=1), val_correct)
reaching = val_thr[val_thr["accepted_accuracy"] >= TARGET_ACCEPTED_ACCURACY]
threshold = float(reaching["threshold"].min()) if len(reaching) else float(THRESHOLDS.max())
print("Chosen confidence threshold:", threshold)
val_thr

*Coverage* = share of queries the system would answer directly. *Accepted accuracy* = accuracy on those queries.
A higher threshold is safer (fewer wrong answers) but asks more users to clarify.

## 5. Final evaluation on the test set (run once)
Metrics and why they matter here:
- **Accuracy**: easy to read, but can hide poor performance on small classes.
- **Macro precision / recall / F1** (primary): every category counts equally.
- **Weighted F1**: overall quality adjusted for class sizes.
- **Per-class recall**: especially `criminal` (must be escalated) and `other_out_of_scope` (must not get a legal answer).
- **Confusion matrix**: shows *which* categories are confused with each other.
- **Per-source results**: show whether the synthetic data makes the task look easier than real data.

In [ ]:
def summary_metrics(y_true, y_pred):
    p_m, r_m, f_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {"n": len(y_true), "accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": p_m, "recall_macro": r_m, "f1_macro": f_m,
            "precision_weighted": p_w, "recall_weighted": r_w, "f1_weighted": f_w}

test_proba = model.predict_proba(test["text"])
test_pred = model.classes_[test_proba.argmax(axis=1)]
test_conf = test_proba.max(axis=1)
y_test = test["label"].to_numpy()

rows = [{"subset": "all_test", **summary_metrics(y_test, test_pred)}]
for source in sorted(test["source"].unique()):
    mask = (test["source"] == source).to_numpy()
    rows.append({"subset": f"test_{source}", **summary_metrics(y_test[mask], test_pred[mask])})
metrics = pd.DataFrame(rows)
metrics.round(4)

In [ ]:
print(classification_report(y_test, test_pred, digits=4, zero_division=0))

In [ ]:
labels = list(model.classes_)
cm = confusion_matrix(y_test, test_pred, labels=labels)
fig, ax = plt.subplots(figsize=(10, 8.5))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("Baseline classifier - confusion matrix (test set, counts)")
plt.tight_layout()
plt.show()

In [ ]:
test_thr = threshold_table(test_conf, test_pred == y_test)
test_thr[test_thr["threshold"] == threshold]

## 6. Looking inside the model
**Top words per category:** the words that push a query most strongly towards each category. This is what makes
the baseline *interpretable*: we can check whether it learned sensible legal vocabulary or accidental shortcuts.

In [ ]:
vocab = np.array(model.named_steps["tfidf"].get_feature_names_out())
clf = model.named_steps["clf"]
weights = clf.coef_ if hasattr(clf, "coef_") else clf.feature_log_prob_
for label, w in zip(labels, weights):
    print(f"{label}: {', '.join(vocab[np.argsort(w)[::-1][:15]])}")

**Some misclassified test examples** (for failure analysis):

In [ ]:
errors = test.assign(predicted=test_pred, confidence=test_conf.round(3))
errors = errors[errors["label"] != errors["predicted"]]
print(f"{len(errors)} of {len(test)} test examples misclassified")
pd.set_option("display.max_colwidth", 120)
errors[["source", "label", "predicted", "confidence", "text"]].head(15)

## 7. Predict function (callable component)
`predict()` takes a raw user query and returns the category, the confidence, the top 3 categories and a
`needs_clarification` flag. This is the interface the retrieval, decision support, recommendation and chatbot
components will use. It can later be wrapped in a FastAPI or Flask endpoint without changes.

In [ ]:
import html, re

def clean_text(text):
    """Same cleaning as src/text_utils.py: strip HTML and URLs, lowercase, collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(re.sub(r"<[^>]+>", " ", text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    return re.sub(r"\s+", " ", text.lower()).strip()

def predict(query):
    text = clean_text(query)
    if not text:
        return {"category": "other_out_of_scope", "confidence": 0.0, "top_3": [], "needs_clarification": True}
    proba = model.predict_proba([text])[0]
    ranked = sorted(zip(model.classes_, proba), key=lambda p: p[1], reverse=True)
    category, confidence = ranked[0]
    return {"category": str(category), "confidence": round(float(confidence), 4),
            "top_3": [(str(c), round(float(p), 4)) for c, p in ranked[:3]],
            "needs_clarification": bool(category == "other_out_of_scope" or confidence < threshold)}

# New queries written for this demo (not taken from the dataset)
demo_queries = [
    "The landlord is refusing to give back the deposit I paid when I moved in.",
    "I have not received my wages from my employer since July.",
    "What do I need to register my new business?",
    "My contract of employment was terminated without notice.",
    "Who will win the election next year?",
    "I have a problem, can you help?",
]
for q in demo_queries:
    print(q, "\n   ->", predict(q), "\n")

## 8. Save the model
The model, threshold and settings are saved together with joblib. `src/predict.py` loads this file.

In [ ]:
model_dir = Path("../models") if Path("../models").exists() else Path("models")
model_dir.mkdir(exist_ok=True)
joblib.dump({"pipeline": model, "threshold": threshold, "labels": labels, "config": best_cfg,
             "sklearn_version": sklearn.__version__}, model_dir / "baseline_classifier.joblib")
print("Saved to", model_dir / "baseline_classifier.joblib")

## 9. Summary
See `docs/interpretation.md` for what these results mean, the limitations of this baseline and the Week 2 plan.

> Kimuntu AI supports user understanding and does not replace professional legal advice.